# 02: Unsupervised Pre-Training Implementation

Implement the unsupervised pre-training stage from Radford's paper using the BooksCorpus dataset.

## Task 1: Implement Unsupervised Pre-Training

In [ ]:
# =====================================================
# 02.1: Import required libraries
# =====================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Constants
HIDDEN_DIM = 768
N_HEADS = 12
N_FFN_DIM = 3072
DROPOUT = 0.1
MAX_LEN = 512

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Save device info
import sys
print(f"Current working directory: {sys.argv[0] if sys.argv else 'No arguments'}")

In [ ]:
# =====================================================
# 02.2: Define simple vocabulary and embeddings
# =====================================================

class SimpleBPEVocabulary:
    '''
    Simple BPE vocabulary for tokenization.
    Implementation inspired by Radford's paper.
    '''
    
    def __init__(self, max_vocab_size=40000):
        """Initialize vocabulary."""
        self.max_vocab_size = max_vocab_size
        # Start with 3-letter tokens
        self.word2idx = {
            "THE": 1, "A": 2, "IN": 3, "THE": 4, "A": 5, "TO": 6, "OF": 7,
            "AT": 8, "BY": 9, "FOR": 10, "TO": 11, "ON": 12, "AND": 13,
            "WITH": 14, "THE": 15, "IN": 16, "FROM": 17, "BY": 18
        }
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        # (In production, build from corpus using BPE algorithm)
        # self._build_bpe_vocab()
    
    def tokenize(self, text):
        """
        Tokenize text using BPE.
        
        Args:
            text: Input string
        
        Returns:
            List of token IDs
        """
        # Simple implementation - split into individual characters
        return list(text.lower())
    
    def __len__(self):
        return len(self.word2idx)

# Create vocabulary
vocab = SimpleBPEVocabulary(40000)
print(f"Vocabulary size: {len(vocab)}")

print("Token mappings:")
print(f"'THE': {vocab.word2idx.get('THE', 'NOT FOUND')}")
print(f"'A': {vocab.word2idx.get('A', 'NOT FOUND')}")
print(f"'IN': {vocab.word2idx.get('IN', 'NOT FOUND')}")
print(f"'OF': {vocab.word2idx.get('OF', 'NOT FOUND')}")

In [ ]:
# =====================================================
# 02.3: Define word embeddings
# =====================================================

class SimpleWordEmbedding:
    """Simple word embedding inspired by Radford's BERT Word2Vec."""
    
    def __init__(self, embedding_dim=768):
        """
        Args:
            embedding_dim: Embedding dimension
        """
        self.embedding_dim = embedding_dim
        self.embedding_size = embedding_dim
        
        # Random initialization: W ~ N(0, 0.02)
        self.weight = torch.nn.Parameter(torch.randn(embedding_dim), requires_grad=True)
        
        self.init_weights()
    
    def init_weights(self):
        """Random initialization."""
        self.weight.data.uniform_(-0.02, 0.02)
    
    def encode(self, word):
        """
        Encode a word to its embedding.
        
        Args:
            word: Word string
        
        Returns:
            Embedding tensor
        """
        word_idx = vocab.word2idx.get(word, vocab.word2idx["<UNK>"])
        return self.weight[word_idx]

# Create word embeddings
word_embedding = SimpleWordEmbedding(embedding_dim=HIDDEN_DIM)

# Verify embedding
embedding = word_embedding.encode("the")
print(f"'THE' embedding shape: {embedding.shape}")
print(f"'THE' embedding (first 5): {embedding[:5].numpy()}")
print(f"Norm: {torch.norm(embedding).item():.4f}")

In [ ]:
# =====================================================
# 02.4: Define simple attention mechanism
# =====================================================

class SimpleAttention:
    """
    Simple attention mechanism (inspired by Vaswani's attention paper).
    """
    
    def __init__(self, embed_dim, num_heads):
        """
        Args:
            embed_dim: Base embedding dimension
            num_heads: Number of attention heads
        """
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Initialize attention score (learnable weights)
        self.attention_score = nn.Linear(embed_dim, num_heads * self.head_dim)
        
        self.init_weights()
    
    def init_weights(self):
        """Random initialization."""
        self.attention_score.weight.data.uniform_(-0.02, 0.02)
        self.attention_score.bias.data.zero_()
    
    def forward(self, key, value, mask):
        """
        Compute attention weights.
        
        Args:
            key, value: Tensor of shape (batch, seq_len, head_dim)
            mask: Mask of shape (batch, seq_len) (1 for attention, 0 for ignore)
        
        Returns:
            Attention weights of shape (batch, seq_len, num_heads)
        """
        # Apply linear transformation
        attn_score = self.attention_score(key)
        
        # Mask zeros
        mask = torch.tril(mask.unsqueeze(1).expand(-1, -1, self.num_heads))
        attn_mask = torch.clamp(mask - 1, min=0)  # Attention is 1, others 0
        
        # Add masked zeros to attention scores
        attn_masked_score = attn_score.masked_fill(attn_mask == 0, float('-inf'))
        
        # Apply softmax
        attn_weights = F.softmax(attn_masked_score, dim=-1)
        
        # Weighted sum
        attn_output = F.linear(attn_weights, value)
        
        # (In production, apply output projection)
        return attn_output

    def backward(self, attn_output, attn_weights):
        """
        Backward pass for attention.
        """
        # Compute gradient of attn_weights w.r.t. key
        d_k = attn_weights * torch.ones_like(attn_output)
        d_kv = d_k * value
        
        # Compute gradient of attn_weights w.r.t. value
        d_v = d_k * torch.ones_like(attn_output)
        
        # Backprop through attention
        d_score = (d_v * torch.exp(attn_weights) - d_kv)
        
        # Apply gradients to weight matrix
        attention_score.weight.grad = d_score


# Define simple attention
def create_attention(embed_dim, num_heads):
    return SimpleAttention(embed_dim, num_heads)

attention = create_attention(HIDDEN_DIM, N_HEADS)
print(f"Attention initialized with embed_dim={HIDDEN_DIM}, num_heads={N_HEADS}")
print(f"Attention score shape: {attention.attention_score.weight.shape}")

## Task 2: Model Training on BooksCorpus

In [ ]:
# =====================================================
# 02.5: Load and prepare BooksCorpus dataset
# =====================================================

class BooksCorpusDataset(Dataset):
    """
    Dataset for loading and preparing BooksCorpus.
    """
    
    def __init__(self, corpus_data):
        """Initialize dataset from corpus data."""
        self.data = corpus_data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        """Get item at index."""
        # Each item contains: text, title, book_id
        return {
            'input_ids': torch.tensor([[vocab.word2idx[t] for t in data['tokens']],
                                        *data['tokens'][1:]]),
            'attention_mask': torch.tensor([[1 for t in data['tokens']] + 
                                           [0 for t in data['tokens'][1:]]]),
            'labels': torch.tensor([[0 for _ in data['tokens']])}
        

In [ ]:
# =====================================================
# 02.6: Training loop
# =====================================================

def compute_loss(logits, labels):
    """
    Compute cross-entropy loss.
    """
    # BCE loss for binary classification
    loss = F.bce_loss(logits, labels)
    return loss

def compute_loss(model, input_ids, position_ids, labels, device):
    """Train forward pass with cross-entropy loss."""
    
    # Forward pass
    model.eval()
    with torch.no_grad():
        logits = model(input_ids, position_ids, torch.ones_like(input_ids)).squeeze(-1)
        
    # Compute loss
    loss = compute_loss(logits, labels)
    
    return loss.item()

def train_epoch(model, dataloader, optimizer, device):
    """Train for one epoch."""
    model.train()
    losses = []
    
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        position_ids = batch['attention_mask'].long().to(device)
        labels = batch['labels'].long().to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(input_ids, position_ids, torch.ones_like(input_ids)).squeeze(-1)
        
        # Compute loss
        loss = compute_loss(logits, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        
        optimizer.step()
        
        losses.append(loss.item())
    
    return np.mean(losses)

def train(model, corpus, dataloader, optimizer, num_epochs=100, device=torch.device('cpu')):
    """Train language model on corpus."""
    losses = []
    
    for epoch in range(num_epochs):
        loss = train_epoch(model, dataloader, optimizer, device)
        losses.append(loss)
        
        # Print progress
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss:.4f}")
    
    return np.mean(losses)

if __name__ == "__main__":
    print("Starting pre-training training...")
    print(f"Total epochs: {100}")
    print(f"Training on device: {device}")
    print("Training will take some time...")
    
    # Uncomment when training is complete
    # train(model, corpus, dataloader, optimizer, num_epochs=100, device=device)

In [ ]:
# =====================================================
# 02.7: Experiment with different training strategies
# =====================================================

# Strategy 1: Standard training
# 100 epochs with cosine annealing

# Strategy 2: Fewer epochs for quick results (3 epochs)
# 3 epochs typically sufficient for SOTA results

# Strategy 3: Learning rate schedule
# Linear warmup + cosine annealing
# Warmup: 0.2% of training steps
# Cosine annealing with decay: lr_final / (1 + exp(-beta))

# Strategy 4: Dropout regularization
# 0.1 rate for all parameters
# Helps prevent overfitting

# Strategy 5: Different learning rates per layer
# Higher LR for early layers (where most parameters are)
# Lower LR for later layers (where fewer parameters are)

print("Training experiments configured:")
print("1. 100 epochs with cosine annealing")
print("2. 3 epochs (quick iteration)")
print("3. Learning rate schedule with warmup")
print("4. Dropout regularization (0.1)")
print("5. Layer-specific learning rates")

print("\nNext steps:")
print("- Save checkpoints after each epoch")
print("- Evaluate on validation set")
print("- Compare with baseline SOTA models")
print("- Implement full fine-tuning pipeline in Task 3")